In [5]:
import pandas as pd
import numpy as np
import re

In [6]:
columns=[
    'area_type',
    'availability',
    'location',
    'size',
    'total_sqft',
    'bath',
    'balcony',
    'price'
]
df=pd.read_csv(
    r"C:\Users\Mahakaal\Documents\HOUSE-PRICE-PREDICTION\data\raw\Bengaluru_House_Data  older.csv",       
    usecols=columns
    )
df.head()

,area_type,availability,location,size,total_sqft,bath,balcony,price
0,Super built-up Area,19-Dec,Electronic City Phase II,2 BHK,1056,2.0,1.0,39.07
1,Plot Area,Ready To Move,Chikka Tirupathi,4 Bedroom,2600,5.0,3.0,120.00
2,Built-up Area,Ready To Move,Uttarahalli,3 BHK,1440,2.0,3.0,62.00
3,Super built-up Area,Ready To Move,Lingadheeranahalli,3 BHK,1521,3.0,1.0,95.00
4,Super built-up Area,Ready To Move,Kothanur,2 BHK,1200,2.0,1.0,51.00


In [7]:
df.info()
df.isnull().sum()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 13320 entries, 0 to 13319
Data columns (total 8 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   area_type     13320 non-null  object 
 1   availability  13320 non-null  object 
 2   location      13319 non-null  object 
 3   size          13304 non-null  object 
 4   total_sqft    13320 non-null  object 
 5   bath          13247 non-null  float64
 6   balcony       12711 non-null  float64
 7   price         13320 non-null  float64
dtypes: float64(3), object(5)
memory usage: 832.6+ KB


area_type         0
availability      0
location          1
size             16
total_sqft        0
bath             73
balcony         609
price             0
dtype: int64

In [8]:
df=df.dropna(subset=['balcony','location'])

df['ready_to_move'] = df['availability'].apply(lambda x: 1 if x == 'Ready To Move' else 0)

df['area_type'] = df['area_type'].astype('category')

df["size"] = df["size"].str.replace("Bedroom", "BHK", regex=False)
df['bedrooms'] = df['size'].str.split().str[0].fillna(0).astype(int)

df['price'] = df['price'] * 100000
df['bath'] = df['bath'].astype('int')

df['balcony'] = df['balcony'].astype('int')
df['location'] = df['location'].str.lower().str.strip()

delete=['availability','size']

df = df.drop(columns=delete)

In [9]:
def clean_sqft_column(x):
    if pd.isna(x) or str(x).strip() == "":
        return np.nan

    val_str = str(x).lower().strip()

    range_match = re.search(r"([0-9.]+)\s*-\s*([0-9.]+)", val_str)
    if range_match:
        try:
            low = float(range_match.group(1))
            high = float(range_match.group(2))
            return (low + high) / 2
        except ValueError:
            return np.nan

    num_match = re.search(r"([0-9.]+)\s*(.*)", val_str)
    if num_match:
        try:
            num = float(num_match.group(1))
            unit = num_match.group(2).strip()

            if not unit:
                return num

            unit_clean = unit.replace(" ", "").replace(".", "").rstrip("s")

            conversion_factors = {
                "sqmeter": 10.7639,
                "sqyard": 9.0,
            }

            if unit_clean in conversion_factors:
                return num * conversion_factors[unit_clean]
            else:
                return np.nan
        except ValueError:
            return np.nan

    return np.nan
df['total_sqft'] = df['total_sqft'].apply(clean_sqft_column)

df = df[(df['total_sqft'] / df['bedrooms']) >= 300]
df = df[df['bath'] <= (df['bedrooms'] + 2)]

In [10]:
df.info()
df.isnull().sum()

<class 'pandas.core.frame.DataFrame'>
Index: 12034 entries, 0 to 13319
Data columns (total 8 columns):
 #   Column         Non-Null Count  Dtype   
---  ------         --------------  -----   
 0   area_type      12034 non-null  category
 1   location       12034 non-null  object  
 2   total_sqft     12034 non-null  float64 
 3   bath           12034 non-null  int64   
 4   balcony        12034 non-null  int64   
 5   price          12034 non-null  float64 
 6   ready_to_move  12034 non-null  int64   
 7   bedrooms       12034 non-null  int64   
dtypes: category(1), float64(2), int64(4), object(1)
memory usage: 764.1+ KB


area_type        0
location         0
total_sqft       0
bath             0
balcony          0
price            0
ready_to_move    0
bedrooms         0
dtype: int64

In [11]:
df.to_csv(r"C:\Users\Mahakaal\Documents\HOUSE-PRICE-PREDICTION\data\proccessed\cleaned_bengaluru_house_data.csv", index=False)